<a href="https://colab.research.google.com/github/gleog6/course/blob/main/mt3/colab/music_transcription_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =================================================================
# 1. SETUP INTEGRATO (ESSENTIA + MAGENTA + MIDO)
# =================================================================
print("Inizializzazione sistema professionale con Essentia...")

# Installazione silenziosa delle dipendenze garantite
!apt-get update -qq && apt-get install -qq libfluidsynth3 build-essential
!pip install -qU "numpy<1.24.0" "typing-extensions<4.6.0"
!pip install -qU essentia pydub magenta soundfile mido

import os
import gc
import numpy as np
import soundfile as sf
import essentia.standard as es
from pydub import AudioSegment
from google.colab import files
import tensorflow.compat.v1 as tf
from mido import MidiFile, MidiTrack

# =================================================================
# 2. MOTORE DI PULIZIA AUDIO (ESSENTIA)
# =================================================================
def apply_essentia_cleaning(samples):
    # Equalizzazione professionale per isolare le fondamentali musicali
    # Taglio drastico delle sub-basse e delle alte frequenze (rumore piatti/sibili)
    eq = es.Equalizer(bands=[31, 62, 125, 250, 500, 1000, 2000, 4000, 8000, 16000],
                      gains=[-25, -10, 2, 3, 3, 3, 2, 0, -15, -30])
    cleaned = eq(samples)

    # Normalizzazione RMS dinamica per mantenere lo stesso volume tra i blocchi
    rms_target = 0.12
    current_rms = np.sqrt(np.mean(cleaned**2))
    return cleaned * (rms_target / (current_rms + 1e-9))

# =================================================================
# 3. CARICAMENTO E PREPARAZIONE BLOCCHI
# =================================================================
uploaded = files.upload()

if not uploaded:
    print("Errore: Seleziona un file per procedere.")
else:
    original_filename = list(uploaded.keys())[0]
    full_audio = AudioSegment.from_file(original_filename).set_frame_rate(16000).set_channels(1)

    # Durata blocco: 3 minuti (180.000 ms)
    chunk_ms = 180000
    chunks = [full_audio[i:i + chunk_ms] for i in range(0, len(full_audio), chunk_ms)]

    print(f"\n[INFO] Brano di {len(full_audio)/60000:.2f} min diviso in {len(chunks)} parti.")

    if not os.path.exists("./checkpoints/mt3"):
        !gsutil -m cp -r gs://mt3/checkpoints .

    generated_midis = []

    # =================================================================
    # 4. LOOP DI TRASCRIZIONE
    # =================================================================
    for idx, chunk in enumerate(chunks):
        print(f"\n>>> Elaborazione Parte {idx+1}/{len(chunks)} (Essentia Engine active)")

        samples = np.array(chunk.get_array_of_samples(), dtype=np.float32)
        samples /= (np.max(np.abs(samples)) + 1e-9)

        # CHIAMATA A ESSENTIA
        samples_cleaned = apply_essentia_cleaning(samples)

        tmp_wav = f"part_{idx}.wav"
        tmp_mid = f"part_{idx}.mid"
        sf.write(tmp_wav, samples_cleaned, 16000)

        tf.reset_default_graph()
        tf.keras.backend.clear_session()

        !python -m magenta.models.mt3.mt3_inference \
            --checkpoint_path="./checkpoints/mt3" \
            --input_audio_path="{tmp_wav}" \
            --output_midi_path="{tmp_mid}"

        if os.path.exists(tmp_mid):
            generated_midis.append(tmp_mid)

        os.remove(tmp_wav)
        gc.collect()

    # =================================================================
    # 5. UNIONE MIDI AUTOMATICA E DOWNLOAD
    # =================================================================
    if generated_midis:
        print("\nAssemblaggio finale dei segmenti MIDI...")
        final_file = "SPARTITO_COMPLETO_ESSENTIA.mid"

        # Unione logica con Mido
        merged = MidiFile()
        track_map = {}

        for m_file in generated_midis:
            m = MidiFile(m_file)
            for track in m.tracks:
                if track.name not in track_map:
                    new_t = MidiTrack()
                    new_t.name = track.name
                    merged.tracks.append(new_t)
                    track_map[track.name] = new_t
                for msg in track:
                    track_map[track.name].append(msg)

        merged.save(final_file)
        print(f"Successo! Scaricamento in corso...")
        files.download(final_file)

        # Pulizia
        for f in generated_midis: os.remove(f)
    else:
        print("Errore critico: Nessun dato MIDI generato.")